<a href="https://colab.research.google.com/github/traceopt-ai/traceml/blob/main/notebooks/huggingface_trl_lora_gradient_accumulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Measure gradient accumulation in a TRL LoRA run

This notebook fine-tunes **Qwen3-1.7B** with **TRL and LoRA** on one T4 GPU. It answers a practical question:

> With the same effective batch size, does a larger per-device batch reduce training time, and how much additional GPU memory does it use?

The two main runs use the same model, data, sequence length, seed, precision, and number of optimizer steps. No Hugging Face login is required.


## The comparison

On one GPU:

```
effective batch = per-device batch x gradient accumulation
```

| Run | Per-device batch | Accumulation | Effective batch |
|---|---:|---:|---:|
| A | 1 | 4 | 4 |
| B | 2 | 2 | 4 |
| C, optional | 4 | 1 | 4 |

Every sample is padded or truncated to 512 tokens, so each optimizer step processes the same number of examples and tokens.

TraceML records one step for each optimizer update. With gradient accumulation, the forward and backward calls from all microbatches are included in that step. Trainer runtime shows the complete training loop, while TraceML shows step timing, phase timing, and peak memory.

Accelerate can move input tensors before the callback opens. For this reason, host-to-device timing may be unavailable in this experiment.


## 1. Set the run length

The default is 30 optimizer steps for each configuration. The third run is optional.


In [ ]:
MAX_STEPS = 100
DATASET_SAMPLES = 256
RUN_THIRD_CASE = True

## 2. Check the GPU

In Colab, select **Runtime > Change runtime type > T4 GPU**.


In [ ]:
import torch

!nvidia-smi -L
assert torch.cuda.is_available(), "Enable a GPU runtime before continuing."
print("Using:", torch.cuda.get_device_name(0))

GPU 0: Tesla T4 (UUID: GPU-25fa23bb-c4af-77b1-12b7-98e903ec9be7)
Using: Tesla T4


## 3. Install dependencies

This experiment uses FP16 LoRA without quantization. Colab may include an older optional `torchao` package that is incompatible with current PEFT, so the first command removes it.


In [ ]:
%pip uninstall -y torchao
%pip install -q -U "traceml-ai[hf]" "trl[peft]" datasets

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 75.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.0/225.0 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.6/559.6 kB 36.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.8/925.8 kB 50.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 66.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.1/82.1 kB 7.9 MB/s eta 0:00:00


## 4. Define the training run

The script uses:

- `Qwen/Qwen3-1.7B`
- `trl-lib/Capybara`
- LoRA rank 16
- sequence length 512
- gradient checkpointing
- FP16 precision

Each configuration runs in a separate process through `traceml run`. This keeps the model and CUDA state isolated between runs.


### Add TraceML

The training code remains a standard `SFTTrainer` run. TraceML needs two additions:

```python
traceml_hf.init()
callbacks=[traceml_hf.TraceMLTrainerCallback()]
```

The notebook writes the complete script below so it remains self-contained.


In [ ]:
%%writefile trl_lora_gradient_accumulation.py
"""Run one TRL LoRA gradient accumulation configuration."""

import argparse
import json
from pathlib import Path

import torch
from datasets import load_dataset
from peft import LoraConfig
from transformers import set_seed
from trl import SFTConfig, SFTTrainer

from traceml_ai.integrations import huggingface as traceml_hf

MODEL_ID = "Qwen/Qwen3-1.7B"
DATASET_ID = "trl-lib/Capybara"
MAX_LENGTH = 512
SEED = 42


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    parser.add_argument("--per-device-batch-size", type=int, required=True)
    parser.add_argument(
        "--gradient-accumulation-steps", type=int, required=True
    )
    parser.add_argument("--max-steps", type=int, default=30)
    parser.add_argument("--dataset-samples", type=int, default=256)
    return parser.parse_args()


def main() -> None:
    args = parse_args()
    if not torch.cuda.is_available():
        raise RuntimeError("This experiment requires a CUDA GPU.")

    set_seed(SEED)

    effective_batch = (
        args.per_device_batch_size * args.gradient_accumulation_steps
    )
    run_name = (
        f"bs{args.per_device_batch_size}_"
        f"ga{args.gradient_accumulation_steps}"
    )
    output_dir = Path("outputs") / run_name
    output_dir.mkdir(parents=True, exist_ok=True)

    print(
        f"Run: {run_name} | "
        f"batch: {args.per_device_batch_size} | "
        f"accumulation: {args.gradient_accumulation_steps} | "
        f"effective batch: {effective_batch}",
        flush=True,
    )

    dataset = load_dataset(DATASET_ID, split="train")
    sample_count = min(args.dataset_samples, len(dataset))
    dataset = dataset.shuffle(seed=SEED).select(range(sample_count))

    lora_config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules="all-linear",
    )

    training_args = SFTConfig(
        output_dir=str(output_dir),
        model_init_kwargs={"dtype": "float16", "use_cache": False},
        per_device_train_batch_size=args.per_device_batch_size,
        gradient_accumulation_steps=args.gradient_accumulation_steps,
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        max_length=MAX_LENGTH,
        pad_to_multiple_of=MAX_LENGTH,
        packing=False,
        max_steps=args.max_steps,
        warmup_steps=5,
        learning_rate=2e-4,
        optim="adamw_torch",
        logging_steps=5,
        save_strategy="no",
        report_to="none",
        disable_tqdm=True,
        fp16=True,
        bf16=False,
        dataloader_num_workers=0,
        dataset_num_proc=1,
        seed=SEED,
        data_seed=SEED,
    )

    traceml_hf.init()
    trainer = SFTTrainer(
        model=MODEL_ID,
        args=training_args,
        train_dataset=dataset,
        peft_config=lora_config,
        callbacks=[traceml_hf.TraceMLTrainerCallback()],
    )

    train_result = trainer.train()
    metrics = {
        **train_result.metrics,
        "run_name": run_name,
        "physical_batch_size": args.per_device_batch_size,
        "gradient_accumulation_steps": (
            args.gradient_accumulation_steps
        ),
        "effective_batch_size": effective_batch,
        "max_length": MAX_LENGTH,
        "optimizer_steps": train_result.global_step,
        "gpu_name": torch.cuda.get_device_name(0),
    }

    metrics_path = output_dir / "trainer_metrics.json"
    metrics_path.write_text(
        json.dumps(metrics, indent=2, sort_keys=True),
        encoding="utf-8",
    )
    print(f"Trainer metrics: {metrics_path}", flush=True)


if __name__ == "__main__":
    main()


Writing trl_lora_gradient_accumulation.py


## 5. Run the two configurations

Run A uses four microbatches per optimizer step. Run B uses two larger microbatches. The first command downloads the model and dataset; the second reuses the local cache. The results table uses training-loop and step metrics, so download time is not included.


In [ ]:
!traceml run \
    --mode summary \
    --logs-dir logs \
    --run-name bs1_ga4 \
    trl_lora_gradient_accumulation.py \
    --args \
    --per-device-batch-size 1 \
    --gradient-accumulation-steps 4 \
    --max-steps {MAX_STEPS} \
    --dataset-samples {DATASET_SAMPLES}


[TraceML] Starting aggregator on 127.0.0.1:29765 (connect=127.0.0.1, ui=summary, profile=run)
[TraceML] Launching TraceML aggregator: /usr/bin/python3 /usr/local/lib/python3.12/dist-packages/traceml_ai/aggregator/aggregator_main.py
[TraceML] Aggregator PID: 1655
[TraceML] Aggregator ready on 127.0.0.1:29765 (workers connect to 127.0.0.1:29765, session=bs1_ga4, ui=summary). Press Ctrl+C to stop.
[TraceML] Aggregator ready.
[TraceML] Launching training process: /usr/bin/python3 -m torch.distributed.run --nnodes=1 --nproc_per_node=1 --node_rank=0 --master_addr=127.0.0.1 --master_port=29500 /usr/local/lib/python3.12/dist-packages/traceml_ai/runtime/executor.py -- --per-device-batch-size 1 --gradient-accumulation-steps 4 --max-steps 100 --dataset-samples 256
Run: bs1_ga4 | batch: 1 | accumulation: 4 | effective batch: 4
README.md: 100% 520/520 [00:00<00:00, 2.54MB/s]

data/train-00000-of-00001.parquet: downloading bytes:  91% 34.0M/37.2M [00:01<00:00, 49.6MB/s, 1.50MB/s  ]
data/train-00000-

In [ ]:
!traceml run \
    --mode summary \
    --logs-dir logs \
    --run-name bs2_ga2 \
    trl_lora_gradient_accumulation.py \
    --args \
    --per-device-batch-size 2 \
    --gradient-accumulation-steps 2 \
    --max-steps {MAX_STEPS} \
    --dataset-samples {DATASET_SAMPLES}


[TraceML] Starting aggregator on 127.0.0.1:29765 (connect=127.0.0.1, ui=summary, profile=run)
[TraceML] Launching TraceML aggregator: /usr/bin/python3 /usr/local/lib/python3.12/dist-packages/traceml_ai/aggregator/aggregator_main.py
[TraceML] Aggregator PID: 3624
[TraceML] Aggregator ready on 127.0.0.1:29765 (workers connect to 127.0.0.1:29765, session=bs2_ga2, ui=summary). Press Ctrl+C to stop.
[TraceML] Aggregator ready.
[TraceML] Launching training process: /usr/bin/python3 -m torch.distributed.run --nnodes=1 --nproc_per_node=1 --node_rank=0 --master_addr=127.0.0.1 --master_port=29500 /usr/local/lib/python3.12/dist-packages/traceml_ai/runtime/executor.py -- --per-device-batch-size 2 --gradient-accumulation-steps 2 --max-steps 100 --dataset-samples 256
Run: bs2_ga2 | batch: 2 | accumulation: 2 | effective batch: 4
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files: 100% 2/2 [00:00<00:00, 1002.46it/s]
Download complete: :           |  0.00B

## 6. Compare the runs

TraceML prints the comparison and saves it as JSON for the results table.


In [ ]:
!traceml compare \
    logs/bs1_ga4/final_summary.json \
    logs/bs2_ga2/final_summary.json \
    --output=logs/bs1_ga4_vs_bs2_ga2


+--------------------------------------------------------------------------------------+
|  TraceML Compare                                                                     |
+--------------------------------------------------------------------------------------+
|                                                                                      |
|  A: bs1_ga4                                                                          |
|  B: bs2_ga2                                                                          |
|  Delta: B - A                                                                        |
|  Primary diagnosis: COMPUTE-BOUND -> COMPUTE-BOUND (same)                            |
|                                                                                      |
|  Verdict: IMPROVEMENT                                                                |
|  Why: GPU Step Time decreased by 10.0%.                                              |
|                    

### Optional: batch 4 without accumulation

Set `RUN_THIRD_CASE = True` above to test a per-device batch of 4. If it does not fit in T4 memory, the notebook reports that no comparison was created.


In [ ]:
from pathlib import Path

if RUN_THIRD_CASE:
    !traceml run --mode summary --logs-dir logs --run-name bs4_ga1 trl_lora_gradient_accumulation.py --args --per-device-batch-size 4 --gradient-accumulation-steps 1 --max-steps {MAX_STEPS} --dataset-samples {DATASET_SAMPLES}

    third_summary = Path("logs/bs4_ga1/final_summary.json")
    if third_summary.exists():
        !traceml compare logs/bs1_ga4/final_summary.json {third_summary} --output=logs/bs1_ga4_vs_bs4_ga1
    else:
        print("Batch 4 did not complete, so no comparison was created.")
else:
    print("Optional run skipped.")

[TraceML] Starting aggregator on 127.0.0.1:29765 (connect=127.0.0.1, ui=summary, profile=run)
[TraceML] Launching TraceML aggregator: /usr/bin/python3 /usr/local/lib/python3.12/dist-packages/traceml_ai/aggregator/aggregator_main.py
[TraceML] Aggregator PID: 5036
[TraceML] Aggregator ready on 127.0.0.1:29765 (workers connect to 127.0.0.1:29765, session=bs4_ga1, ui=summary). Press Ctrl+C to stop.
[TraceML] Aggregator ready.
[TraceML] Launching training process: /usr/bin/python3 -m torch.distributed.run --nnodes=1 --nproc_per_node=1 --node_rank=0 --master_addr=127.0.0.1 --master_port=29500 /usr/local/lib/python3.12/dist-packages/traceml_ai/runtime/executor.py -- --per-device-batch-size 4 --gradient-accumulation-steps 1 --max-steps 100 --dataset-samples 256
Run: bs4_ga1 | batch: 4 | accumulation: 1 | effective batch: 4
Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files: 100% 2/2 [00:00<00:00, 3452.10it/s]
Download complete: :           |  0.00B

## 7. Review the result

The table combines Trainer runtime with TraceML step timing and memory measurements.


In [ ]:
import json
from pathlib import Path

import pandas as pd


def load_json(path):
    with Path(path).open(encoding="utf-8") as handle:
        return json.load(handle)


def compare_value(payload, section, metric, side):
    return (
        payload.get("sections", {})
        .get(section, {})
        .get("metrics", {})
        .get(metric, {})
        .get(side)
    )


def rounded(value, digits=2, scale=1.0):
    if value is None:
        return None
    return round(float(value) / scale, digits)


def result_row(run_name, batch_size, accumulation, comparison, side):
    trainer = load_json(f"outputs/{run_name}/trainer_metrics.json")
    return {
        "run": run_name,
        "batch": batch_size,
        "accumulation": accumulation,
        "effective batch": batch_size * accumulation,
        "Trainer runtime (s)": rounded(trainer.get("train_runtime")),
        "Trainer steps/s": rounded(
            trainer.get("train_steps_per_second"), digits=3
        ),
        "TraceML step (ms)": rounded(
            compare_value(comparison, "step_time", "step_time_ms", side)
        ),
        "forward (ms)": rounded(
            compare_value(comparison, "step_time", "forward_ms", side)
        ),
        "backward (ms)": rounded(
            compare_value(comparison, "step_time", "backward_ms", side)
        ),
        "peak reserved (GiB)": rounded(
            compare_value(
                comparison,
                "step_memory",
                "peak_reserved_bytes",
                side,
            ),
            scale=1024**3,
        ),
    }


primary = load_json("logs/bs1_ga4_vs_bs2_ga2.json")
rows = [
    result_row("bs1_ga4", 1, 4, primary, "lhs"),
    result_row("bs2_ga2", 2, 2, primary, "rhs"),
]

optional_path = Path("logs/bs1_ga4_vs_bs4_ga1.json")
if RUN_THIRD_CASE and optional_path.exists():
    optional = load_json(optional_path)
    rows.append(result_row("bs4_ga1", 4, 1, optional, "rhs"))

pd.DataFrame(rows)

,run,batch,accumulation,effective batch,Trainer runtime (s),Trainer steps/s,TraceML step (ms),forward (ms),backward (ms),peak reserved (GiB)
0,bs1_ga4,1,4,4,287.57,0.348,2870.85,986.34,1765.05,5.45
1,bs2_ga2,2,2,4,258.78,0.386,2584.72,835.12,1674.78,5.58
2,bs4_ga1,4,1,4,238.21,0.420,2379.15,778.72,1547.30,5.81


## Read the table

- Trainer runtime and steps per second show the overall training-loop difference.
- TraceML step, forward, and backward times show where that difference occurred.
- Peak reserved memory shows the cost of using a larger per-device batch.
- Similar values mean that changing the microbatch layout had little effect on this setup.
- Missing host-to-device timing does not mean that data transfer took no time. It may occur outside the traced callback window.

For benchmark numbers, run each configuration three times in fresh runtimes and report the median.

To apply the same check to another trainer, keep the model, data order, sequence length, precision, effective batch, and optimizer-step count fixed. Change only the per-device batch and gradient accumulation.
